In [57]:
RANDOM_STATE = 42
OUT_DIR = "runs"
RUN_NAME = "elliptic"

In [58]:
# =========================================
# 0. Import & cấu hình chung
# =========================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from scipy.stats import randint, uniform
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.base import clone
from scipy.stats import randint, uniform
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.metrics import roc_auc_score


# Thư mục chứa các file Elliptic (sửa lại cho đúng máy bạn)
DATA_DIR = r"D:\elliptic\Elliptic_Dataset"
TXS_FEATURES_FILE = "txs_features.csv"
TXS_CLASSES_FILE  = "txs_classes.csv"
TXS_EDGELIST_FILE = "txs_edgelist.csv"  # nếu cần cho GNN thì dùng thêm, ở đây chưa cần

In [59]:
# =========================================
# 1. Load dataset
# =========================================
import os

df_txs_features = pd.read_csv(os.path.join(DATA_DIR, TXS_FEATURES_FILE))
df_txs_classes  = pd.read_csv(os.path.join(DATA_DIR, TXS_CLASSES_FILE))

print("txs_features shape:", df_txs_features.shape)
print("txs_classes  shape:", df_txs_classes.shape)
print("txs_features columns:", df_txs_features.columns.tolist())
print("txs_classes  columns:", df_txs_classes.columns.tolist())


txs_features shape: (203769, 184)
txs_classes  shape: (203769, 2)
txs_features columns: ['txId', 'Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10', 'Local_feature_11', 'Local_feature_12', 'Local_feature_13', 'Local_feature_14', 'Local_feature_15', 'Local_feature_16', 'Local_feature_17', 'Local_feature_18', 'Local_feature_19', 'Local_feature_20', 'Local_feature_21', 'Local_feature_22', 'Local_feature_23', 'Local_feature_24', 'Local_feature_25', 'Local_feature_26', 'Local_feature_27', 'Local_feature_28', 'Local_feature_29', 'Local_feature_30', 'Local_feature_31', 'Local_feature_32', 'Local_feature_33', 'Local_feature_34', 'Local_feature_35', 'Local_feature_36', 'Local_feature_37', 'Local_feature_38', 'Local_feature_39', 'Local_feature_40', 'Local_feature_41', 'Local_feature_42', 'Local_feature_43', 'Local_feature_44', 'Local_feature_45',

In [60]:
# =========================================
# 2. Merge & xử lý label
# =========================================
# Giả định:
#   - df_txs_features có cột 'txId' và 'Time step' (hoặc 'time_step')
#   - df_txs_classes có cột 'txId' và 'class'
#   - class:
#       1 = illicit
#       2 = licit
#       3 = unknown
# Ta drop class=3, và tạo label nhị phân: 0=licit, 1=illicit

df = df_txs_features.merge(df_txs_classes, on="txId", how="left")

if "class" not in df.columns:
    raise ValueError("Không tìm thấy cột 'class' sau khi merge!")

print("\nPhân bố class ban đầu:")
print(df["class"].value_counts(dropna=False))

# Bỏ các giao dịch không có nhãn hoặc nhãn 'unknown' = 3
df = df.dropna(subset=["class"]).copy()
df["class"] = df["class"].astype(int)
df = df[df["class"] != 3].copy()

# Tạo label nhị phân: 0=licit, 1=illicit
df["label"] = (df["class"] == 2).astype(int)

LABEL_COL = "label"
print("\nPhân bố label sau khi bỏ unknown (0=illicit,1=licit):")
print(df[LABEL_COL].value_counts())


Phân bố class ban đầu:
class
3    157205
2     42019
1      4545
Name: count, dtype: int64

Phân bố label sau khi bỏ unknown (0=illicit,1=licit):
label
1    42019
0     4545
Name: count, dtype: int64


In [61]:
# =========================================
# 3. Chọn feature (loại id/time/label)
# =========================================
# Cột id / thời gian không dùng làm feature
possible_ts_cols = ["Time step", "time_step"]
ts_col = None
for c in possible_ts_cols:
    if c in df.columns:
        ts_col = c
        break

if ts_col is None:
    raise ValueError("Không tìm thấy cột time-step (ví dụ 'Time step' hoặc 'time_step')!")

drop_id_cols = ["txId", ts_col]
drop_label_cols = ["class", LABEL_COL]

cols_to_drop = [c for c in drop_id_cols + drop_label_cols if c in df.columns]

feature_cols = [c for c in df.columns if c not in cols_to_drop]

X_df = df[feature_cols].copy()
y = df[LABEL_COL].values
time_steps = df[ts_col].values

print("\nSố feature:", len(feature_cols))
print("Một vài feature đầu:", feature_cols[:10])

# Chỉ giữ các cột số
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
X_df = X_df[num_cols].copy()
print("\nSố feature numeric:", len(num_cols))


Số feature: 182
Một vài feature đầu: ['Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10']

Số feature numeric: 182


In [62]:
# =========================================
# 4. Chia train/val/test THEO TIME_STEP (30/10/10)
# =========================================
unique_ts = np.sort(df[ts_col].unique())
n_ts = len(unique_ts)
print("\nSố time-step khác nhau:", n_ts)
print("Các time-step đầu:", unique_ts[:10], "...", unique_ts[-10:])

# Nếu đúng Elliptic (49 time-step), ta dùng 30/10/9
# Nếu >= 50, dùng đúng 30/10/10
# Nếu ít hơn thì chia theo tỉ lệ 60/20/20
if n_ts == 49:
    train_ts = unique_ts[:34]
    val_ts   = unique_ts[34:41]
    test_ts  = unique_ts[41:]
elif n_ts >= 50:
    train_ts = unique_ts[:30]
    val_ts   = unique_ts[30:40]
    test_ts  = unique_ts[40:50]
else:
    # fallback: chia theo tỉ lệ tương đối
    idx_train_end = int(0.6 * n_ts)
    idx_val_end   = int(0.8 * n_ts)
    train_ts = unique_ts[:idx_train_end]
    val_ts   = unique_ts[idx_train_end:idx_val_end]
    test_ts  = unique_ts[idx_val_end:]

print("\nTime-step TRAIN:", train_ts[0], "->", train_ts[-1])
print("Time-step VAL  :", val_ts[0],   "->", val_ts[-1])
print("Time-step TEST :", test_ts[0],  "->", test_ts[-1])

train_mask = df[ts_col].isin(train_ts)
val_mask   = df[ts_col].isin(val_ts)
test_mask  = df[ts_col].isin(test_ts)

X_train = X_df[train_mask].values
y_train = y[train_mask]

X_val   = X_df[val_mask].values
y_val   = y[val_mask]

X_test  = X_df[test_mask].values
y_test  = y[test_mask]

print("\nKích thước:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

print("\nPhân bố nhãn train:")
print(pd.Series(y_train).value_counts())
print("\nPhân bố nhãn val:")
print(pd.Series(y_val).value_counts())
print("\nPhân bố nhãn test:")
print(pd.Series(y_test).value_counts())


Số time-step khác nhau: 49
Các time-step đầu: [ 1  2  3  4  5  6  7  8  9 10] ... [40 41 42 43 44 45 46 47 48 49]

Time-step TRAIN: 1 -> 34
Time-step VAL  : 35 -> 41
Time-step TEST : 42 -> 49

Kích thước:
X_train: (29894, 182) y_train: (29894,)
X_val  : (7829, 182) y_val  : (7829,)
X_test : (8841, 182) y_test : (8841,)

Phân bố nhãn train:
1    26432
0     3462
Name: count, dtype: int64

Phân bố nhãn val:
1    7154
0     675
Name: count, dtype: int64

Phân bố nhãn test:
1    8433
0     408
Name: count, dtype: int64


In [63]:
# =========================================
# 5. Chuẩn hóa feature (fit trên TRAIN)
# =========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)


In [64]:
# =========================================
# 6. AutoML đơn giản: random search trên VAL (macro-F1)
#    Không dùng k-fold
# =========================================
def sample_param(dist, rng):
    """Lấy 1 giá trị từ distribution hoặc list."""
    if hasattr(dist, "rvs"):
        # scipy distribution
        return dist.rvs(random_state=rng)
    # list / tuple / set ...
    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

def random_search_single_model(
    name,
    base_estimator,
    param_dist,
    X_train, y_train,
    X_val,   y_val,
    n_iter=30,
    scoring="macro"
):
    """
    Random search đơn giản:
      - Mỗi iter: sample 1 bộ siêu tham số
      - Train trên TRAIN
      - Đánh giá trên VAL theo F1-macro
      - Chọn bộ tốt nhất, rồi train lại trên TRAIN+VAL với bộ đó
    """
    print(f"\n===== Random search cho {name} (không k-fold, dùng VAL) =====")
    rng = np.random.RandomState(RANDOM_STATE)
    best_f1 = -1.0
    best_params = None

    for i in range(n_iter):
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = clone(base_estimator)
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        # macro-F1 dùng để chọn model
        f1_macro = f1_score(y_val, y_val_pred, average="macro")

        print(f"Iter {i+1:02d}/{n_iter}: F1_macro(val) = {f1_macro:.4f}, params = {params}")

        if f1_macro > best_f1:
            best_f1 = f1_macro
            best_params = params

    print(f"\n>>> {name} – best F1_macro(val) = {best_f1:.4f}")
    print("Best params:", best_params)

    # Train lại trên TRAIN+VAL với best_params trước khi test
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_model = clone(base_estimator)
    best_model.set_params(**best_params)
    best_model.fit(X_train_full, y_train_full)

    return best_model

# Tính scale_pos_weight cho XGB/LGBM
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
print("\nscale_pos_weight (train):", scale_pos_weight)


scale_pos_weight (train): 0.130977602905569


In [65]:
# =========================================
# 7. Unsupervised AutoML: Autoencoder + Isolation Forest
#    - Tune trên VAL (dùng nhãn VAL để chọn hyperparams + threshold)
#    - Sau tune: fit lại trên TRAIN+VAL (chỉ dùng mẫu normal) để test
# =========================================

from dataclasses import dataclass
from typing import Dict, Any, Optional
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from sklearn.base import clone
import numpy as np
import pandas as pd


# -------------------------------------------------
# 0) Thiết lập nhãn normal/anomaly (theo code hiện tại của bạn)
# -------------------------------------------------
# Theo cell trước: label = (class == 2).astype(int)
# => 1 = licit (normal), 0 = illicit (anomaly)
NORMAL_LABEL = 1
ANOMALY_LABEL = 0

print("NORMAL_LABEL =", NORMAL_LABEL, "| ANOMALY_LABEL =", ANOMALY_LABEL)
print("Train label counts:\n", pd.Series(y_train).value_counts().sort_index())
print("Val label counts:\n", pd.Series(y_val).value_counts().sort_index())
print("Test label counts:\n", pd.Series(y_test).value_counts().sort_index())


NORMAL_LABEL = 1 | ANOMALY_LABEL = 0
Train label counts:
 0     3462
1    26432
Name: count, dtype: int64
Val label counts:
 0     675
1    7154
Name: count, dtype: int64
Test label counts:
 0     408
1    8433
Name: count, dtype: int64


In [66]:
# -------------------------------------------------
# 1) Utilities
# -------------------------------------------------
def sample_param(dist, rng):
    """Lấy 1 giá trị từ distribution hoặc list/tuple."""
    if hasattr(dist, "rvs"):  # scipy dist
        return dist.rvs(random_state=rng)
    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

def to_binary_anomaly(y, anomaly_label=ANOMALY_LABEL):
    """
    Convert nhãn gốc -> nhãn đánh giá anomaly:
      1 = anomaly
      0 = normal
    """
    return (y == anomaly_label).astype(int)

def metric_on_val(y_val_raw, y_val_pred_raw):
    """
    Đánh giá theo bài toán anomaly:
      - anomaly là positive class (1 sau convert)
    """
    y_true_bin = to_binary_anomaly(y_val_raw)
    y_pred_bin = to_binary_anomaly(y_val_pred_raw)
    f1_macro = f1_score(y_true_bin, y_pred_bin, average="macro")
    f1_anom  = f1_score(y_true_bin, y_pred_bin, average="binary", pos_label=1)
    return f1_macro, f1_anom

def threshold_search(scores_val, y_val_raw, mode="higher", n_grid=200):
    """
    Tìm threshold tốt nhất trên VAL.
    - mode='higher': score cao hơn => anomaly
    - mode='lower' : score thấp hơn => anomaly
    Trả về threshold + metric tốt nhất.
    """
    y_true_bin = to_binary_anomaly(y_val_raw)

    scores_val = np.asarray(scores_val).reshape(-1)
    if len(np.unique(scores_val)) == 1:
        # score hằng -> fallback
        thr = scores_val[0]
        if mode == "higher":
            y_pred_bin = (scores_val >= thr).astype(int)
        else:
            y_pred_bin = (scores_val <= thr).astype(int)
        return {
            "threshold": float(thr),
            "f1_macro": f1_score(y_true_bin, y_pred_bin, average="macro"),
            "f1_anom":  f1_score(y_true_bin, y_pred_bin, average="binary", pos_label=1, zero_division=0),
        }

    # Grid threshold theo quantile để ổn định hơn
    qs = np.linspace(0.001, 0.999, n_grid)
    thrs = np.unique(np.quantile(scores_val, qs))

    best = {"threshold": None, "f1_macro": -1.0, "f1_anom": -1.0}

    for thr in thrs:
        if mode == "higher":
            y_pred_bin = (scores_val >= thr).astype(int)
        else:
            y_pred_bin = (scores_val <= thr).astype(int)

        f1_macro = f1_score(y_true_bin, y_pred_bin, average="macro", zero_division=0)
        f1_anom  = f1_score(y_true_bin, y_pred_bin, average="binary", pos_label=1, zero_division=0)

        # ưu tiên macro-F1, tie-break bằng F1 anomaly
    # thay điều kiện chọn best trong threshold_search bằng:
    if (f1_anom > best["f1_anom"]) or (np.isclose(f1_anom, best["f1_anom"]) and f1_macro > best["f1_macro"]):
        best = {"threshold": float(thr), "f1_macro": float(f1_macro), "f1_anom": float(f1_anom)}

    return best



In [67]:
# -------------------------------------------------
# 2) Base wrapper classes (fit unsupervised trên normal samples)
# -------------------------------------------------
class IFAnomalyWrapper:
    """
    Isolation Forest wrapper:
    - fit(X, y): chỉ fit trên normal samples
    - anomaly score cao hơn => bất thường hơn
    """
    def __init__(self, **kwargs):
        self.model = IsolationForest(**kwargs)

    def set_params(self, **params):
        self.model.set_params(**params)
        return self

    def get_params(self, deep=True):
        return self.model.get_params(deep=deep)

    def fit(self, X, y=None):
        if y is None:
            X_fit = X
        else:
            X_fit = X[y == NORMAL_LABEL]   # cực kỳ quan trọng
        self.model.fit(X_fit)
        return self

    def score_samples_for_anomaly(self, X):
        # decision_function: lớn hơn = normal hơn
        # đảo dấu để lớn hơn = anomaly hơn
        return -self.model.decision_function(X)

    def predict_with_threshold(self, X, threshold):
        scores = self.score_samples_for_anomaly(X)
        y_pred_anom = (scores >= threshold).astype(int)  # 1 = anomaly
        y_pred_raw = np.where(y_pred_anom == 1, ANOMALY_LABEL, NORMAL_LABEL)
        return y_pred_raw, scores


class AutoencoderMLPWrapper:
    """
    Autoencoder dùng MLPRegressor (X -> X):
    - fit(X, y): chỉ fit trên normal samples
    - anomaly score = reconstruction MSE (cao hơn => bất thường hơn)
    """
    def __init__(self, **kwargs):
        self.model = MLPRegressor(**kwargs)

    def set_params(self, **params):
        self.model.set_params(**params)
        return self

    def get_params(self, deep=True):
        return self.model.get_params(deep=deep)

    def fit(self, X, y=None):
        if y is None:
            X_fit = X
        else:
            X_fit = X[y == NORMAL_LABEL]
        self.model.fit(X_fit, X_fit)
        return self

    def score_samples_for_anomaly(self, X):
        X_hat = self.model.predict(X)
        mse = np.mean((X - X_hat) ** 2, axis=1)
        return mse

    def predict_with_threshold(self, X, threshold):
        scores = self.score_samples_for_anomaly(X)
        y_pred_anom = (scores >= threshold).astype(int)  # 1=anomaly
        y_pred_raw = np.where(y_pred_anom == 1, ANOMALY_LABEL, NORMAL_LABEL)
        return y_pred_raw, scores

In [68]:
# =========================================
# Patch NaN cho dữ liệu scaled (trước unsupervised)
# =========================================
from sklearn.impute import SimpleImputer
import numpy as np

print("NaN counts BEFORE impute:")
print("train:", np.isnan(X_train_scaled).sum())
print("val  :", np.isnan(X_val_scaled).sum())
print("test :", np.isnan(X_test_scaled).sum())

# Fit imputer trên TRAIN (tránh leak test)
# Nếu muốn chặt hơn cho anomaly detection: fit trên train-normal
# X_imputer_fit = X_train_scaled[y_train == NORMAL_LABEL]
X_imputer_fit = X_train_scaled

imputer_unsup = SimpleImputer(strategy="median")
imputer_unsup.fit(X_imputer_fit)

X_train_scaled_imp = imputer_unsup.transform(X_train_scaled)
X_val_scaled_imp   = imputer_unsup.transform(X_val_scaled)
X_test_scaled_imp  = imputer_unsup.transform(X_test_scaled)

print("\nNaN counts AFTER impute:")
print("train:", np.isnan(X_train_scaled_imp).sum())
print("val  :", np.isnan(X_val_scaled_imp).sum())
print("test :", np.isnan(X_test_scaled_imp).sum())

NaN counts BEFORE impute:
train: 3315
val  : 3774
test : 1734

NaN counts AFTER impute:
train: 0
val  : 0
test : 0


In [69]:

# -------------------------------------------------
# 3) Hàm random search cho unsupervised model
# -------------------------------------------------
def random_search_unsupervised(
    name,
    model_builder,        # hàm trả về wrapper mới
    param_dist,           # dict distributions/lists
    X_train, y_train,
    X_val, y_val,
    n_iter=20,
    random_state=RANDOM_STATE
):
    """
    Quy trình mỗi iter:
      1) sample params
      2) fit trên TRAIN (chỉ normal)
      3) tính anomaly score trên VAL
      4) tune threshold trên VAL
      5) lấy metric tốt nhất
    Sau cùng:
      - fit lại model với best_params trên TRAIN+VAL (chỉ normal)
      - threshold giữ nguyên (từ VAL tuning)
    """
    print(f"\n===== Random search UNSUPERVISED cho {name} (tune trên VAL) =====")
    rng = np.random.RandomState(random_state)

    best = {
        "f1_macro": -1.0,
        "f1_anom": -1.0,
        "params": None,
        "threshold": None,
        "model": None,
    }

    for i in range(n_iter):
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = model_builder()
        model.set_params(**params)
        model.fit(X_train, y_train)

        scores_val = model.score_samples_for_anomaly(X_val)
        thr_info = threshold_search(scores_val, y_val, mode="higher", n_grid=200)

        print(
            f"Iter {i+1:02d}/{n_iter} | "
            f"F1_macro(val)={thr_info['f1_macro']:.4f} | "
            f"F1_anom(val)={thr_info['f1_anom']:.4f} | "
            f"thr={thr_info['threshold']:.6f} | params={params}"
        )

        better = (
            (thr_info["f1_macro"] > best["f1_macro"]) or
            (np.isclose(thr_info["f1_macro"], best["f1_macro"]) and thr_info["f1_anom"] > best["f1_anom"])
        )
        if better:
            best["f1_macro"] = thr_info["f1_macro"]
            best["f1_anom"]  = thr_info["f1_anom"]
            best["params"]   = params
            best["threshold"] = thr_info["threshold"]

    print(f"\n>>> {name} best on VAL:")
    print("    F1_macro(val) =", round(best["f1_macro"], 4))
    print("    F1_anom(val)  =", round(best["f1_anom"], 4))
    print("    threshold     =", best["threshold"])
    print("    params        =", best["params"])

    # --- Fit FINAL trên TRAIN+VAL (chỉ normal) ---
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    final_model = model_builder()
    final_model.set_params(**best["params"])
    final_model.fit(X_train_full, y_train_full)

    best["model"] = final_model
    return best



In [70]:
# -------------------------------------------------
# 4) Không gian hyperparams (AutoML random search)
# -------------------------------------------------
# Isolation Forest
if_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_samples":  ["auto", 256, 512, 1024],
    "max_features": [0.5, 0.7, 1.0],
    "bootstrap":    [False, True],
    "contamination": ["auto"],   # threshold sẽ tune riêng trên VAL
    "random_state": [RANDOM_STATE],
    "n_jobs":       [-1],
}

# Autoencoder (MLPRegressor dạng shallow/deep vừa phải)
ae_param_dist = {
    "hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32)],
    "activation": ["relu", "tanh"],
    "solver": ["adam"],
    "alpha": [1e-5, 1e-4, 1e-3],
    "learning_rate_init": [1e-4, 5e-4, 1e-3],
    "batch_size": [128, 256, "auto"],
    "max_iter": [80, 120, 160],
    "early_stopping": [True],
    "validation_fraction": [0.1],
    "n_iter_no_change": [10, 15],
    "random_state": [RANDOM_STATE],
}


In [71]:
# -------------------------------------------------
# 5) Chạy AutoML cho từng model
# -------------------------------------------------
if_result = random_search_unsupervised(
    name="IsolationForest",
    model_builder=lambda: IFAnomalyWrapper(),
    param_dist=if_param_dist,
    X_train=X_train_scaled, y_train=y_train,
    X_val=X_val_scaled, y_val=y_val,
    n_iter=20,   # tăng lên 30-50 nếu muốn
    random_state=RANDOM_STATE
)


===== Random search UNSUPERVISED cho IsolationForest (tune trên VAL) =====
Iter 01/20 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=0.143803 | params={'n_estimators': 300, 'max_samples': 1024, 'max_features': 0.5, 'bootstrap': False, 'contamination': 'auto', 'random_state': 42, 'n_jobs': -1}
Iter 02/20 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=0.143803 | params={'n_estimators': 300, 'max_samples': 1024, 'max_features': 0.5, 'bootstrap': False, 'contamination': 'auto', 'random_state': 42, 'n_jobs': -1}
Iter 03/20 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=0.169925 | params={'n_estimators': 300, 'max_samples': 256, 'max_features': 1.0, 'bootstrap': False, 'contamination': 'auto', 'random_state': 42, 'n_jobs': -1}
Iter 04/20 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=0.154958 | params={'n_estimators': 300, 'max_samples': 512, 'max_features': 0.5, 'bootstrap': True, 'contamination': 'auto', 'random_state': 42, 'n_jobs': -1}
Iter 05/20 | F1_macro(val)=0.4772 |

In [72]:
print("=== DEBUG LABEL MAPPING ===")
print("NORMAL_LABEL =", NORMAL_LABEL, "| ANOMALY_LABEL =", ANOMALY_LABEL)

# số lượng trong train/val/test
for name, y_ in [("train", y_train), ("val", y_val), ("test", y_test)]:
    vc = pd.Series(y_).value_counts().sort_index()
    print(f"{name} label counts:\n{vc}\n")

# check số lượng normal trong train để fit IF
n_normal_train = np.sum(y_train == NORMAL_LABEL)
n_anom_train   = np.sum(y_train == ANOMALY_LABEL)
print("train normals used for IF fit:", n_normal_train)
print("train anomalies:", n_anom_train)

=== DEBUG LABEL MAPPING ===
NORMAL_LABEL = 1 | ANOMALY_LABEL = 0
train label counts:
0     3462
1    26432
Name: count, dtype: int64

val label counts:
0     675
1    7154
Name: count, dtype: int64

test label counts:
0     408
1    8433
Name: count, dtype: int64

train normals used for IF fit: 26432
train anomalies: 3462


In [78]:
# -------------------------------------------------
# 6) Đánh giá TEST (sau khi đã fit FINAL trên TRAIN+VAL)
# -------------------------------------------------
def evaluate_unsupervised_result(name, result, X_test, y_test):
    model = result["model"]
    thr   = result["threshold"]

    y_pred_raw, scores_test = model.predict_with_threshold(X_test, thr)

    # Metrics theo nhãn gốc
    acc = accuracy_score(y_test, y_pred_raw)
    f1_macro_raw = f1_score(y_test, y_pred_raw, average="macro")
    f1_micro_raw = f1_score(y_test, y_pred_raw, average="micro")
    f1_weighted_raw = f1_score(y_test, y_pred_raw, average="weighted")

    # Metrics theo anomaly-positive (1 = anomaly sau convert)
    y_test_bin = to_binary_anomaly(y_test)
    y_pred_bin = to_binary_anomaly(y_pred_raw)
    f1_anom = f1_score(y_test_bin, y_pred_bin, average="binary", pos_label=1, zero_division=0)
    f1_macro_anom = f1_score(y_test_bin, y_pred_bin, average="macro", zero_division=0)

    print(f"\n================ {name} | TEST ================")
    print("Best params :", result["params"])
    print("Threshold   :", thr)
    print(f"Accuracy    : {acc:.4f}")
    print(f"F1 macro(raw labels)     : {f1_macro_raw:.4f}")
    print(f"F1 micro(raw labels)     : {f1_micro_raw:.4f}")
    print(f"F1 weighted(raw labels)  : {f1_weighted_raw:.4f}")
    print(f"F1 anomaly(binary pos=anomaly) : {f1_anom:.4f}")
    print(f"F1 macro(binary anomaly task)  : {f1_macro_anom:.4f}")

    print("\nConfusion matrix (raw labels, rows=true, cols=pred):")
    print(confusion_matrix(y_test, y_pred_raw, labels=[ANOMALY_LABEL, NORMAL_LABEL]))

    print("\nClassification report (raw labels):")
    print(classification_report(
        y_test, y_pred_raw,
        labels=[ANOMALY_LABEL, NORMAL_LABEL],
        target_names=[f"anomaly(label={ANOMALY_LABEL})", f"normal(label={NORMAL_LABEL})"],
        zero_division=0
    ))
        # =========================================================
    # AUC-ROC (quan trọng: scores_test cao hơn = bất thường hơn)
    # Dataset của bạn: 0 = anomaly, 1 = normal
    # => tạo nhãn tạm y_test_anom01 để anomaly là positive class (1)
    # =========================================================
    y_test_arr = np.asarray(y_test)

    # anomaly-positive label cho AUC
    y_test_anom01 = (y_test_arr == ANOMALY_LABEL).astype(int)

    # AUC cho phát hiện anomaly (đúng hướng score: cao = anomaly)
    try:
        auc_roc_anomaly = roc_auc_score(y_test_anom01, scores_test)
    except ValueError:
        # xảy ra nếu test chỉ có 1 lớp
        auc_roc_anomaly = np.nan

    # (optional) AUC nếu muốn nhìn theo normal là positive
    # score_normal = -scores_test vì điểm thấp hơn thường "normal" hơn
    y_test_normal01 = (y_test_arr == NORMAL_LABEL).astype(int)
    try:
        auc_roc_normal = roc_auc_score(y_test_normal01, -scores_test)
    except ValueError:
        auc_roc_normal = np.nan
    print(f"auc_roc_norma : {auc_roc_normal:.4f}")
    print(f"auc_roc_anomaly : {auc_roc_anomaly:.4f}")
    return {
        "name": name,
        "acc": acc,
        "f1_macro_raw": f1_macro_raw,
        "f1_weighted_raw": f1_weighted_raw,
        "f1_anomaly": f1_anom,
        "f1_macro_anom": f1_macro_anom,
        "auc_roc_anomaly": auc_roc_anomaly,   # <-- thêm
        "auc_roc_normal": auc_roc_normal,     # <-- thêm (optional)
        "threshold": thr,
        "params": result["params"],
        "scores_test": scores_test,
        "y_pred_raw": y_pred_raw
    }

res_if = evaluate_unsupervised_result("IsolationForest", if_result, X_test_scaled, y_test)


================ IsolationForest | TEST ================
Best params : {'n_estimators': 300, 'max_samples': 1024, 'max_features': 0.5, 'bootstrap': False, 'contamination': 'auto', 'random_state': 42, 'n_jobs': -1}
Threshold   : 0.1438026213097038
Accuracy    : 0.9535
F1 macro(raw labels)     : 0.4881
F1 micro(raw labels)     : 0.9535
F1 weighted(raw labels)  : 0.9312
F1 anomaly(binary pos=anomaly) : 0.0000
F1 macro(binary anomaly task)  : 0.4881

Confusion matrix (raw labels, rows=true, cols=pred):
[[   0  408]
 [   3 8430]]

Classification report (raw labels):
                  precision    recall  f1-score   support

anomaly(label=0)       0.00      0.00      0.00       408
 normal(label=1)       0.95      1.00      0.98      8433

        accuracy                           0.95      8841
       macro avg       0.48      0.50      0.49      8841
    weighted avg       0.91      0.95      0.93      8841

auc_roc_norma : 0.2814
auc_roc_anomaly : 0.2814


In [74]:
print("Pred counts IF on test:", pd.Series(res_if["y_pred_raw"]).value_counts().sort_index())
print("True counts test       :", pd.Series(y_test).value_counts().sort_index())

Pred counts IF on test: 0       3
1    8838
Name: count, dtype: int64
True counts test       : 0     408
1    8433
Name: count, dtype: int64


In [75]:
ae_result = random_search_unsupervised(
    name="Autoencoder-MLP",
    model_builder=lambda: AutoencoderMLPWrapper(),
    param_dist=ae_param_dist,
    X_train=X_train_scaled_imp, y_train=y_train,
    X_val=X_val_scaled_imp, y_val=y_val,
    n_iter=15,   # AE train lâu hơn IF
    random_state=RANDOM_STATE
)


===== Random search UNSUPERVISED cho Autoencoder-MLP (tune trên VAL) =====
Iter 01/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=6.518188 | params={'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 10, 'random_state': 42}


d:\elliptic\venv1\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(


Iter 02/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=1.543998 | params={'hidden_layer_sizes': (128,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.001, 'batch_size': 'auto', 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}
Iter 03/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=13.598835 | params={'hidden_layer_sizes': (256,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 1e-05, 'learning_rate_init': 0.0005, 'batch_size': 256, 'max_iter': 120, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}


d:\elliptic\venv1\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(


Iter 04/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=6.297776 | params={'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'solver': 'adam', 'alpha': 1e-05, 'learning_rate_init': 0.0005, 'batch_size': 256, 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}
Iter 05/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=7.776526 | params={'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.001, 'batch_size': 'auto', 'max_iter': 120, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}
Iter 06/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=25.621398 | params={'hidden_layer_sizes': (128, 64), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.0005, 'batch_size': 256, 'max_iter': 160, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}


...

Iter 10/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=6.979743 | params={'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.001, 'batch_size': 256, 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}


d:\elliptic\venv1\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


Iter 11/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=9.151284 | params={'hidden_layer_sizes': (128,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate_init': 0.0005, 'batch_size': 256, 'max_iter': 120, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}
Iter 12/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=27.743370 | params={'hidden_layer_sizes': (128, 64), 'activation': 'tanh', 'solver': 'adam', 'alpha': 1e-05, 'learning_rate_init': 0.001, 'batch_size': 256, 'max_iter': 120, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}
Iter 13/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=9.245181 | params={'hidden_layer_sizes': (128,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate_init': 0.0005, 'batch_size': 'auto', 'max_iter': 160, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 15, 'random_state': 42}
Iter 

d:\elliptic\venv1\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (80) reached and the optimization hasn't converged yet.
  warnings.warn(


Iter 15/15 | F1_macro(val)=0.4772 | F1_anom(val)=0.0000 | thr=25.409642 | params={'hidden_layer_sizes': (256, 128), 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0001, 'learning_rate_init': 0.0001, 'batch_size': 128, 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 10, 'random_state': 42}

>>> Autoencoder-MLP best on VAL:
    F1_macro(val) = 0.4772
    F1_anom(val)  = 0.0
    threshold     = 6.518188444926815
    params        = {'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 10, 'random_state': 42}


In [79]:
res_ae = evaluate_unsupervised_result("Autoencoder-MLP", ae_result, X_test_scaled_imp, y_test)


================ Autoencoder-MLP | TEST ================
Best params : {'hidden_layer_sizes': (128, 64), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.001, 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 80, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 10, 'random_state': 42}
Threshold   : 6.518188444926815
Accuracy    : 0.9532
F1 macro(raw labels)     : 0.4904
F1 micro(raw labels)     : 0.9532
F1 weighted(raw labels)  : 0.9312
F1 anomaly(binary pos=anomaly) : 0.0048
F1 macro(binary anomaly task)  : 0.4904

Confusion matrix (raw labels, rows=true, cols=pred):
[[   1  407]
 [   7 8426]]

Classification report (raw labels):
                  precision    recall  f1-score   support

anomaly(label=0)       0.12      0.00      0.00       408
 normal(label=1)       0.95      1.00      0.98      8433

        accuracy                           0.95      8841
       macro avg       0.54      0.50      0.49      8841
    weighted avg       0.92      